In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("BronzeJaffleShopOrders").getOrCreate()

SOURCE_FILE_PATH = "/Volumes/workspace/default/raw_data/jaffle_shop/orders.csv"
TARGET_TABLE_NAME = "workspace.default.bronze_jaffle_shop_orders"

# HYPOTHETICAL schema — confirm with source system owner before production use
orders_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("status", StringType(), True),
])

try:
    source_df = spark.read.csv(
        SOURCE_FILE_PATH,
        header=True,
        schema=orders_schema,
        sep=",",
        multiLine=False,
    )

    bronze_df = (
        source_df
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    bronze_df.write.format("delta")         .mode("overwrite")         .saveAsTable(TARGET_TABLE_NAME)

    print(f"Successfully loaded data from {SOURCE_FILE_PATH} to {TARGET_TABLE_NAME}")
    print(f"Records written: {bronze_df.count()}")

except Exception as e:
    import traceback
    print(f"Error during ETL process: {e}")
    traceback.print_exc()


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, trim, current_timestamp, to_timestamp, count
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

spark = SparkSession.builder.appName("OrdersDQValidation").getOrCreate()

# HYPOTHETICAL schema for orders.csv — must be confirmed with source system owner
customer_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("status", StringType(), True),
])

# Sample data including various data quality issues
sample_data = [
    (1, 1, "2023-01-15", "completed"),
    (2, 2, "2023-02-20", "shipped"),
    (3, None, "2023-03-01", "pending"),   # Missing user_id
    (4, 3, None, "placed"),               # Missing order_date
    (5, 4, "2023-05-10", None),           # Missing status
    (None, 5, "2023-06-15", "completed"), # Missing id
    (1, 6, "2023-07-20", "completed"),    # Duplicate id
]

df_orders = spark.createDataFrame(sample_data, ["id", "user_id", "order_date", "status"])

validation_rules = [
    {
        "name": "order_id_not_null",
        "purpose": "Every order must have a unique identifier.",
        "logic": col("id").isNull(),
        "severity": "Critical",
        "message": "Order ID is missing.",
        "type": "row_level",
        "column": "id",
    },
    {
        "name": "user_id_not_null",
        "purpose": "Every order must be linked to a user.",
        "logic": col("user_id").isNull(),
        "severity": "Critical",
        "message": "User ID is missing.",
        "type": "row_level",
        "column": "user_id",
    },
    {
        "name": "order_date_not_null",
        "purpose": "Order date is required for fulfilment tracking.",
        "logic": col("order_date").isNull(),
        "severity": "High",
        "message": "Order date is missing.",
        "type": "row_level",
        "column": "order_date",
    },
    {
        "name": "status_not_null_or_empty",
        "purpose": "Order status is required for pipeline routing.",
        "logic": col("status").isNull() | (trim(col("status")) == ""),
        "severity": "High",
        "message": "Order status is missing or empty.",
        "type": "row_level",
        "column": "status",
    },
    {
        "name": "order_id_unique",
        "purpose": "Order IDs must be unique.",
        "logic_column": ["id"],
        "severity": "Critical",
        "message": "Duplicate Order ID found.",
        "type": "aggregate_level",
        "column": "id",
    },
]


def run_validations(df, rules):
    """Run row-level and aggregate DQ rules; return DataFrame of failed records."""
    failed_records_dfs = []
    output_schema_fields = df.schema.fields + [
        StructField("_dq_rule_name", StringType(), True),
        StructField("_dq_severity", StringType(), True),
        StructField("_dq_failure_message", StringType(), True),
    ]
    output_schema = StructType(output_schema_fields)

    for rule in [r for r in rules if r["type"] == "row_level"]:
        failed_df = df.filter(rule["logic"])
        if failed_df.count() > 0:
            failed_records_dfs.append(
                failed_df
                .withColumn("_dq_rule_name", lit(rule["name"]))
                .withColumn("_dq_severity", lit(rule["severity"]))
                .withColumn("_dq_failure_message", lit(rule["message"]))
            )

    for rule in [r for r in rules if r["type"] == "aggregate_level"]:
        group_cols = rule["logic_column"]
        duplicate_values_df = (
            df.groupBy(*group_cols)
            .agg(count(lit(1)).alias("_cnt"))
            .filter(col("_cnt") > 1)
            .select(*group_cols)
        )
        if duplicate_values_df.count() > 0:
            failed_df = df.join(duplicate_values_df, on=group_cols, how="inner")
            failed_records_dfs.append(
                failed_df
                .withColumn("_dq_rule_name", lit(rule["name"]))
                .withColumn("_dq_severity", lit(rule["severity"]))
                .withColumn("_dq_failure_message", lit(rule["message"]))
            )

    if not failed_records_dfs:
        return spark.createDataFrame([], schema=output_schema)

    union_df = failed_records_dfs[0]
    for fdf in failed_records_dfs[1:]:
        union_df = union_df.unionByName(fdf, allowMissingColumns=True)
    return union_df.distinct()


failed_df = run_validations(df_orders, validation_rules)
print("--- Failed Order Records ---")
if failed_df.count() > 0:
    failed_df.show(truncate=False)
else:
    print("No order records failed data quality validations.")

# Data Quality Dimensions Checklist:
# Completeness:  [x]
# Uniqueness:    [x]
# Validity:      [x]
# Accuracy:      [ ]
# Consistency:   [ ]
# Integrity:     [ ]
# Timeliness:    [ ]
# Volume:        [ ]
